# Demo00 · 用 LLM 生成旅行计划

**这个 Notebook 的唯一目标**：在不使用任何 Agent 框架的前提下，理解 LLM API 调用的完整机制。

你将学到：
1. LLM API 的本质：一次 HTTP 请求
2. `messages` 是什么，`role` 有哪些，各自的作用
3. 如何让模型返回结构化 JSON 而不是自由文本
4. 如何解析并使用模型的输出

> **为什么从这里开始？**  
> LangChain、LangGraph、AutoGen 等所有框架，本质上都在封装这几步操作。
> 先把底层看透，框架就是工具而不是黑箱。

## 第一步：安装依赖

- `openai`：OpenAI 官方 Python SDK，也支持任意兼容 OpenAI 协议的模型（DeepSeek、通义、Moonshot 等）
- `python-dotenv`：从 `.env` 文件加载环境变量，避免把密钥写死在代码里

In [1]:
%pip install openai python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


## 第二步：配置 API 密钥

在同目录下创建一个 `.env` 文件（参考 `.env.example`），内容如下：

```
OPENAI_API_KEY=sk-xxxxxx
OPENAI_BASE_URL=https://api.openai.com/v1   # 如果用国内代理或 DeepSeek，改这里
OPENAI_MODEL=gpt-4o-mini
```

如果没有 API Key，先跳到最后一节，可以直接看模拟输出理解流程。

In [2]:
import json
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

API_KEY   = os.getenv("OPENAI_API_KEY", "")
BASE_URL  = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
MODEL     = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

# 创建客户端 —— 它只是帮你管理 base_url 和 headers，HTTP 请求还是它发的
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

print(f"Model : {MODEL}")
print(f"Key   : {'已配置 ✓' if API_KEY else '未配置，将使用 fallback 演示'}")

Model : gpt-4o-mini
Key   : 已配置 ✓


## 第三步：理解 messages 格式

调用 LLM 最核心的概念是 **messages 列表**，每条消息有两个字段：

| role | 含义 | 类比 |
|------|------|------|
| `system` | 给模型的「角色设定」或「行为规则」 | 入职培训手册 |
| `user` | 用户这一轮发送的内容 | 用户当前的提问 |
| `assistant` | 模型上一轮的回复（多轮对话时放入） | 模型之前说过的话 |

**关键点**：模型没有记忆，每次调用都要把完整的对话历史塞进 `messages`。
这就是为什么长对话会消耗大量 token。

## 第四步：第一次调用 —— 自由文本输出

In [3]:
messages_v1 = [
    {
        "role": "system",
        "content": "你是一名专业的中文旅行规划师，擅长设计实用、有节奏感的行程。",
    },
    {
        "role": "user",
        "content": "帮我规划一个北京 3 天的旅行行程，偏好历史文化，预算中等。",
    },
]

response_v1 = client.chat.completions.create(
    model=MODEL,
    messages=messages_v1,
    temperature=0.7,   # 0=确定性输出，1=更有创意，旅行规划 0.7 是合理区间
)

print(response_v1.choices[0].message.content)

当然可以！以下是一个为期3天的北京历史文化之旅行程，预算中等，适合您深入体验这座城市的独特魅力。

### 第一天：故宫与天安门广场

**上午：**
- **天安门广场**  
  参观中国的象征——天安门广场。这里有人民英雄纪念碑和毛主席纪念堂，可以了解中国现代历史。
  
- **故宫博物院**  
  从天安门广场步行至故宫，建议提前在线购票。故宫是明清两代的皇宫，规模宏大，建筑风格独特。建议租用讲解器或参加导游讲解以更好地理解其历史。

**午餐：**  
在故宫附近的“东来顺”享用正宗的北京涮羊肉，体验当地美食。

**下午：**
- **景山公园**  
  从故宫北门出发，步行至景山公园，登顶可俯瞰故宫全景，是拍照的好地方。

**晚上：**  
- **前门大街**  
  前往前门大街，体验老北京的风情，选择一家餐馆品尝北京烤鸭，如“全聚德”或“便宜坊”。

### 第二天：长城与明十三陵

**上午：**
- **长城（八达岭或慕田峪）**  
  提前预定好包车或参加一日游，建议选择慕田峪长城，游客相对较少，风景优美。可选择缆车上山，节省体力。

**午餐：**  
在长城附近的农家乐享用当地特色午餐。

**下午：**
- **明十三陵**  
  从长城前往明十三陵，参观长陵或其他陵墓，了解明代皇帝的历史和文化。

**晚上：**  
返回市区，选择在“南锣鼓巷”附近的小吃街，尝试各种北京小吃。

### 第三天：天坛与胡同文化

**上午：**
- **天坛公园**  
  参观天坛，了解古代皇帝的祭天文化，公园内环境优美，非常适合晨练和散步。

**午餐：**  
在天坛附近的小吃店享用早餐或午餐。

**下午：**
- **南锣鼓巷与胡同游**  
  探索南锣鼓巷的特色小店，体验北京的胡同文化。可以选择骑自行车或步行游览，感受老北京的生活气息。

**晚上：**  
- **后海酒吧街**  
  在后海附近享用晚餐，选择一家有特色的餐厅，结束您的北京之行。可以在后海边散步，欣赏夜景。

### 旅行小贴士：
- **交通**：建议使用地铁和打车，方便快捷。
- **预算**：根据个人消费习惯，预计每餐约50-100元，景点门票和交通费另算，整体预算可控制在1500-2500元/人。
- **天气**：请关注天气预报，适时调整行程

## 第五步：观察完整的响应结构

模型返回的不只是文字，还有很多元数据，理解它们很重要：

In [5]:
print("=== 完整响应对象 ===")
print(response_v1)
print()
print("=== 关键字段解析 ===")
print(f"模型名称    : {response_v1.model}")
print(f"消耗 token  : prompt={response_v1.usage.prompt_tokens}, "
      f"completion={response_v1.usage.completion_tokens}, "
      f"total={response_v1.usage.total_tokens}")
print(f"停止原因    : {response_v1.choices[0].finish_reason}")
# finish_reason = 'stop'  → 正常结束
# finish_reason = 'length' → 被 max_tokens 截断，内容不完整！

=== 完整响应对象 ===
ChatCompletion(id='chatcmpl-DOdUyXEH3pgHgRbPToV8MLmQoEosq', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='当然可以！以下是一个为期3天的北京历史文化之旅行程，预算中等，适合您深入体验这座城市的独特魅力。\n\n### 第一天：故宫与天安门广场\n\n**上午：**\n- **天安门广场**  \n  参观中国的象征——天安门广场。这里有人民英雄纪念碑和毛主席纪念堂，可以了解中国现代历史。\n  \n- **故宫博物院**  \n  从天安门广场步行至故宫，建议提前在线购票。故宫是明清两代的皇宫，规模宏大，建筑风格独特。建议租用讲解器或参加导游讲解以更好地理解其历史。\n\n**午餐：**  \n在故宫附近的“东来顺”享用正宗的北京涮羊肉，体验当地美食。\n\n**下午：**\n- **景山公园**  \n  从故宫北门出发，步行至景山公园，登顶可俯瞰故宫全景，是拍照的好地方。\n\n**晚上：**  \n- **前门大街**  \n  前往前门大街，体验老北京的风情，选择一家餐馆品尝北京烤鸭，如“全聚德”或“便宜坊”。\n\n### 第二天：长城与明十三陵\n\n**上午：**\n- **长城（八达岭或慕田峪）**  \n  提前预定好包车或参加一日游，建议选择慕田峪长城，游客相对较少，风景优美。可选择缆车上山，节省体力。\n\n**午餐：**  \n在长城附近的农家乐享用当地特色午餐。\n\n**下午：**\n- **明十三陵**  \n  从长城前往明十三陵，参观长陵或其他陵墓，了解明代皇帝的历史和文化。\n\n**晚上：**  \n返回市区，选择在“南锣鼓巷”附近的小吃街，尝试各种北京小吃。\n\n### 第三天：天坛与胡同文化\n\n**上午：**\n- **天坛公园**  \n  参观天坛，了解古代皇帝的祭天文化，公园内环境优美，非常适合晨练和散步。\n\n**午餐：**  \n在天坛附近的小吃店享用早餐或午餐。\n\n**下午：**\n- **南锣鼓巷与胡同游**  \n  探索南锣鼓巷的特色小店，体验北京的胡同

## 第六步：结构化输出 —— JSON 模式

自由文本输出有个致命问题：**代码无法可靠地解析它**。

解决方案：在调用时加上 `response_format={"type": "json_object"}`，
同时在 `system` prompt 里明确告诉模型你需要的 JSON 结构。

> **原理**：这会开启模型的「JSON 约束解码」模式，保证输出是合法的 JSON 字符串。
> 注意：只保证语法合法，不保证字段完整——所以 system prompt 里的结构说明很重要。

In [9]:
SYSTEM_PROMPT_JSON = """
你是一名专业的中文旅行规划师。
请严格以 JSON 格式返回行程，不要包含任何 JSON 之外的文字。

JSON 结构如下：
{
  "city": "目的地城市",
  "travel_days": 天数(整数),
  "summary": "整体行程概述",
  "days": [
    {
      "day": 第几天(整数),
      "theme": "当天主题",
      "morning": "上午安排",
      "afternoon": "下午安排",
      "evening": "晚上安排",
      "meals": ["早餐建议", "午餐建议", "晚餐建议"],
      "tips": ["当天小贴士"]
    }
  ],
  "packing_tips": ["打包建议"],
  "budget_advice": "预算建议"
}
"""

messages_v2 = [
    {"role": "system", "content": SYSTEM_PROMPT_JSON},
    {
        "role": "user",
        "content": "帮我规划一个深圳 3 天的旅行行程，偏好潮流时尚，预算中等。",
    },
]

response_v2 = client.chat.completions.create(
    model=MODEL,
    messages=messages_v2,
    temperature=0.7,
    response_format={"type": "json_object"},  # 关键：开启 JSON 模式
)

raw_json_str = response_v2.choices[0].message.content
print("模型原始输出（字符串）：")
print(raw_json_str[:200], "...")  # 先看前200字符

模型原始输出（字符串）：
{
  "city": "深圳",
  "travel_days": 3,
  "summary": "在深圳的三天行程中，您将探索时尚潮流的购物区，体验现代艺术和文化，同时享受深圳的美食。",
  "days": [
    {
      "day": 1,
      "theme": "时尚购物体验",
      "morning": "前往深圳湾公园散步，享受清晨的海风",
      ...


## 第七步：解析 JSON 并使用数据

In [10]:
trip_plan = json.loads(raw_json_str)

# 现在 trip_plan 是一个普通的 Python dict，可以直接用
print(f"目的地   : {trip_plan['city']}")
print(f"行程天数 : {trip_plan['travel_days']} 天")
print(f"概述     : {trip_plan['summary']}")
print()

for day in trip_plan["days"]:
    print(f"{'='*50}")
    print(f"第 {day['day']} 天 · {day['theme']}")
    print(f"  上午：{day['morning']}")
    print(f"  下午：{day['afternoon']}")
    print(f"  晚上：{day['evening']}")
    if day.get("meals"):
        print(f"  餐饮：{' | '.join(day['meals'])}")
    if day.get("tips"):
        for tip in day["tips"]:
            print(f"  💡 {tip}")

print(f"{'='*50}")
print(f"\n打包建议：")
for t in trip_plan.get("packing_tips", []):
    print(f"  - {t}")
print(f"\n预算建议：{trip_plan.get('budget_advice', '')}")

目的地   : 深圳
行程天数 : 3 天
概述     : 在深圳的三天行程中，您将探索时尚潮流的购物区，体验现代艺术和文化，同时享受深圳的美食。

第 1 天 · 时尚购物体验
  上午：前往深圳湾公园散步，享受清晨的海风
  下午：游览购物公园（Shopping Park），体验国际品牌和潮流店铺
  晚上：在购物公园附近的餐厅享用晚餐，之后前往南山区的酒吧街
  餐饮：酒店早餐 | 购物公园的快餐 | 酒吧街的特色小吃
  💡 提前查询购物公园的折扣信息，合理安排行程
第 2 天 · 艺术与文化
  上午：参观OCT创意文化园，欣赏现代艺术展览
  下午：继续在OCT周边的创意商店和咖啡馆游览
  晚上：在OCT附近的餐馆享用晚餐，体验当地的夜生活
  餐饮：酒店早餐 | OCT内的艺术风格餐厅 | 当地特色餐厅
  💡 带上相机记录艺术作品，注意园区内的活动时间
第 3 天 · 科技与未来
  上午：参观深圳博物馆，了解深圳的科技发展
  下午：前往前海湾，体验未来城市的建设和风貌
  晚上：在前海湾附近的海鲜餐厅享用晚餐，欣赏海景
  餐饮：酒店早餐 | 博物馆附近的快餐 | 前海湾的海鲜餐厅
  💡 关注深圳博物馆的免费开放时间，避免高峰期

打包建议：
  - 轻便舒适的鞋子
  - 适合拍照的时尚服装
  - 太阳镜和防晒霜

预算建议：建议每天的预算在500-800元之间，包含餐饮、交通和购物


## 第八步：封装成可复用的函数

上面的代码走通了，现在把它整理成一个函数——这就是 `demo01/app/llm_client.py` 的雏形。

In [11]:
def plan_trip(
    city: str,
    travel_days: int,
    preferences: list[str] | None = None,
    budget_level: str = "medium",
    notes: str = "",
) -> dict:
    """调用 LLM，返回结构化的旅行计划 dict。"""
    prefs = "、".join(preferences) if preferences else "城市漫游、当地美食"
    notes_text = f"额外要求：{notes}" if notes else ""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_JSON},
        {
            "role": "user",
            "content": (
                f"帮我规划一个 {city} {travel_days} 天的旅行行程。"
                f"偏好：{prefs}。预算：{budget_level}。{notes_text}"
            ),
        },
    ]

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.7,
        response_format={"type": "json_object"},
    )

    return json.loads(response.choices[0].message.content)


# 试一下
result = plan_trip(
    city="成都",
    travel_days=2,
    preferences=["川菜美食", "熊猫基地"],
    budget_level="低预算",
)

print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "city": "成都",
  "travel_days": 2,
  "summary": "本次成都之行以熊猫基地和川菜美食为主题，适合预算有限的旅行者。",
  "days": [
    {
      "day": 1,
      "theme": "熊猫基地与川菜探索",
      "morning": "前往成都大熊猫繁育研究基地，观看可爱的熊猫",
      "afternoon": "在熊猫基地附近的餐厅享用川菜，推荐小吃如冷吃兔",
      "evening": "游览锦里古街，体验当地文化与美食",
      "meals": [
        "豆浆油条",
        "冷吃兔",
        "串串香"
      ],
      "tips": [
        "早上尽量早点到熊猫基地，避开人流高峰"
      ]
    },
    {
      "day": 2,
      "theme": "文化与美食",
      "morning": "参观武侯祠，了解三国历史",
      "afternoon": "在附近的餐馆尝试担担面，体验地道风味",
      "evening": "在春熙路附近享受夜市美食",
      "meals": [
        "包子",
        "担担面",
        "糖油果子"
      ],
      "tips": [
        "注意保持水分，成都天气较干燥"
      ]
    }
  ],
  "packing_tips": [
    "轻便舒适的衣物",
    "防晒霜",
    "水瓶"
  ],
  "budget_advice": "尽量选择当地餐馆和街边小吃，避免高档餐厅，适当控制购物预算"
}


## 第九步：没有 API Key？用 fallback 演示数据理解结构

如果你暂时没有 API Key，下面这段代码模拟了模型应该返回的数据结构，
让你能跑通后续的解析逻辑。

In [ ]:
MOCK_RESPONSE = {
    "city": "北京",
    "travel_days": 3,
    "summary": "围绕历史文化设计的三日行程，节奏适中，覆盖核心地标与胡同生活。",
    "days": [
        {
            "day": 1,
            "theme": "故宫·天安门·景山",
            "morning": "天安门广场观升旗，随后进入故宫，重点游览太和殿、乾清宫、珍宝馆。",
            "afternoon": "登景山俯瞰故宫全景，步行至南锣鼓巷感受胡同文化。",
            "evening": "什刹海边吃烤鸭，夜游后海。",
            "meals": ["早餐：天安门附近早点铺", "午餐：故宫内御膳餐厅", "晚餐：南锣鼓巷烤鸭店"],
            "tips": ["故宫需提前在官网预约门票，周一闭馆", "升旗时间随季节变化，提前查询"],
        },
        {
            "day": 2,
            "theme": "长城·颐和园",
            "morning": "早出发前往慕田峪长城，上午人少，风景最佳。",
            "afternoon": "返程后游览颐和园，沿昆明湖漫步，参观佛香阁。",
            "evening": "五道口或中关村附近晚餐，体验北京年轻人的生活。",
            "meals": ["早餐：便利店或酒店", "午餐：慕田峪长城景区餐厅", "晚餐：五道口特色餐厅"],
            "tips": ["长城建议穿运动鞋，带足水", "颐和园下午光线好，适合拍照"],
        },
        {
            "day": 3,
            "theme": "天坛·798·收尾",
            "morning": "天坛公园，欣赏祈年殿，可看到老人打太极、下象棋。",
            "afternoon": "798 艺术区逛展览、咖啡、当代艺术，轻松收尾。",
            "evening": "王府井步行街购物，采购北京特产后前往机场或火车站。",
            "meals": ["早餐：天坛附近老北京早点", "午餐：798 文艺餐厅", "晚餐：王府井小吃街"],
            "tips": ["天坛早上有大量本地居民晨练，是了解北京市民生活的好机会"],
        },
    ],
    "packing_tips": ["舒适运动鞋（长城必备）", "充电宝", "防晒霜和折叠伞"],
    "budget_advice": "中等预算三天约 1500-2500 元，主要开销在交通、故宫/长城门票和餐饮。",
}

# 用和真实调用完全相同的解析逻辑处理 mock 数据
for day in MOCK_RESPONSE["days"]:
    print(f"第 {day['day']} 天 · {day['theme']}")
    print(f"  上午：{day['morning'][:40]}...")
print("\n[使用 Mock 数据演示成功]")

## 总结：你刚刚理解了什么

```
你的代码
  │
  ├─ 构造 messages=[{role, content}, ...]   ← 你控制的输入
  │
  ├─ POST https://api.openai.com/v1/chat/completions
  │     model, temperature, response_format  ← 你调的旋钮
  │
  └─ 解析 response.choices[0].message.content  ← 模型的输出
        │
        └─ json.loads() → Python dict → 你的业务逻辑
```

**所有框架都只是这个流程的封装**：

| 框架 | 它帮你封装了什么 |
|------|-----------------|
| `langchain` | 消息模板、多模型切换、链式调用 |
| `langgraph` | 多步骤有状态的流程编排（图结构） |
| `autogen` | 多 Agent 之间的对话协作 |

---

### 下一步：demo01

`demo01` 把这个函数包进了 FastAPI，加上了 Pydantic schema 做输入校验，
并提供了一个前端页面——这就是「主链路闭环」的最小可用版本。

去看 `demo01/app/llm_client.py`，你会发现它和本 Notebook 的逻辑完全一致，
只是换成了 `httpx` 异步调用（`async/await`）以便配合 FastAPI。